# Phase 4 — Train model_25m on Kaggle TPU v5e-8

**Before you start:**
1. Mount the `llm-forge-tokens-v1` dataset from the Kaggle sidebar (+ Add data → search `adeshboudh/llm-forge-tokens-v1`).
2. Run the cells in order. Cells 1-3 set up the environment; cell 4+ run training.

In [ ]:
!git clone https://github.com/adeshboudh/llm_forge.git 2>/dev/null || (cd llm_forge && git pull)

In [ ]:
!pip install uv && cd llm_forge && uv sync --extra dev

In [ ]:
!cd llm_forge && uv run python -c "import jax, flax, optax, orbax.checkpoint; print('jax:', jax.__version__); print('devices:', jax.devices())"

In [ ]:
!ls /kaggle/input/datasets/adeshboudh/llm-forge-tokens-v1/ | head -5 && echo "---" && ls /kaggle/input/datasets/adeshboudh/llm-forge-tokens-v1/ | wc -l && echo "shards above (expect 216: 215 npy + metadata.json)"

In [ ]:
!cd llm_forge && uv run python -m training.summary --config configs/training/model_25m.yaml

In [ ]:
# Sanity: 50 steps only, confirms TPU detected + loss decreases.
!cd llm_forge && uv run python -m training.train --config configs/training/model_25m.yaml --max-steps 50

In [ ]:
# Full 1B-token run (9766 steps, ~3-6 hours on 8 v5e cores).
!cd llm_forge && uv run python -m training.train --config configs/training/model_25m.yaml

In [ ]:
import json
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open('/kaggle/working/train_log.jsonl')]
steps = [r['step'] for r in rows]
losses = [r['loss'] for r in rows]
val = [(r['step'], r['val_loss']) for r in rows if r.get('val_loss') is not None]

plt.plot(steps, losses, label='train')
if val:
    plt.plot(*zip(*val), 'o-', label='val')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.show()

In [ ]:
# Load final checkpoint and generate 3 samples (smoke; real sampling in Phase 6).
import jax, jax.numpy as jnp
from model.config import load_model_config
from model.lm import LM
from training.state import create_train_state, restore

cfg = load_model_config('model_25m')
model = LM(config=cfg)
state = create_train_state(jax.random.PRNGKey(0), model, None, cfg)  # placeholder
# state = restore('/kaggle/working/ckpt/step_000009766_final', state)  # uncomment after run

prompt = jnp.array([[3, 4, 5, 6]], dtype=jnp.int32)  # <|bos|> + 3 random tokens
loss, logits = model.apply(state.params, prompt, prompt, return_logits=True)
print('logits shape:', logits.shape, '(full sampling lives in Phase 6)')